## Batch Georeference a Subset of Photos

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
from datetime import datetime, timezone
from tqdm.auto import tqdm
import shutil

from single_photo_workflow import (
    download_photo,
    get_image_metadata,
    calculate_horizontal_extent,
    update_tracker
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

### Prepare the subset and batch

In [ ]:
SUBSET_PATH = Path(
    "data/historic_airphoto_index/subsets/"
    "subset_20000_carver_37_38_40_51_57_63.gpkg"
)

LAYER = "photos"

# to process the next batch, change this to 100, then 200, etc. 
START = 0
BATCH_SIZE = 100

In [ ]:
# Create a separate GeoPackage for tracking processing progress
TRACKER_PATH = SUBSET_PATH.with_name(
    f"{SUBSET_PATH.stem}_tracker.gpkg"
)

# Only create it if it doesn't already exist
if not TRACKER_PATH.exists():
    shutil.copy2(SUBSET_PATH, TRACKER_PATH)
    print(f"Created tracker: {TRACKER_PATH.name}")
else:
    print(f"Using existing tracker: {TRACKER_PATH.name}")

In [ ]:
gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

In [ ]:
# Use the GeoPackage name as the downloads folder
DOWNLOAD_DIR = Path("data/downloads") / SUBSET_PATH.stem
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Stable order for repeatable chunking
gdf = gdf.sort_values("cty_year_id").reset_index(drop=True)

In [ ]:
batch_gdf = gdf.iloc[START:START + BATCH_SIZE].copy()

print(f"Selected {len(batch_gdf)} records")
print(f"Downloads: {DOWNLOAD_DIR}")

In [ ]:
batch_gdf

### Download JPEG, read metadata, calculate extent

In [ ]:
results = []

for _, photo in tqdm(
    batch_gdf.iterrows(),
    total=len(batch_gdf),
    desc="Preparing photos"
):
    photo_id = photo["cty_year_id"]

    image_path = DOWNLOAD_DIR / f"{photo_id}_original.jpg"

    result = {
        "source_id": photo["source_id"],
        "cty_year_id": photo_id,
        "width_px": None,
        "dpi_x": None,
        "h_extent": None,
        "date_download": None,
        "download_status": "pending",
        "error": None,
    }

    try:
        image_path = download_photo(
            url=photo["url"],
            output_path=image_path,
        )

        result["download_status"] = "complete"

        result["date_download"] = datetime.now(
            timezone.utc
        ).isoformat()

        metadata = get_image_metadata(image_path)

        result.update(metadata)

        result["h_extent"] = calculate_horizontal_extent(
            width_px=metadata["width_px"],
            dpi_x=metadata["dpi_x"],
            scale=photo["scale"],
        )

    except Exception as e:
        result["error"] = str(e)

    results.append(result)

In [ ]:
results_df = pd.DataFrame(results)

# Keep the original batch with all its fields and geometry
prepared_gdf = batch_gdf.copy()

# Match results to records using source_id
updates = results_df.set_index("source_id")

# Update only successful values
for column in ["width_px", "dpi_x", "h_extent", "date_download"]:
    values = prepared_gdf["source_id"].map(updates[column])
    prepared_gdf[column] = values.combine_first(prepared_gdf[column])

# Update status only for fully prepared photos
successful_ids = set(
    results_df.loc[
        results_df["error"].isna(), "source_id"
    ]
)

prepared_gdf.loc[
    prepared_gdf["source_id"].isin(successful_ids),
    "status"
] = "prepared"

display(prepared_gdf.head())

In [ ]:
updated = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated} records in tracker.")

In [ ]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

display(
    tracker_gdf[
        ["cty_year_id", "status", "width_px",
         "dpi_x", "h_extent", "date_download"]
    ].head()
)